<h1 style="color: #CEDDF4;">DEPI Round 4, MS Data Engineer and AI Track</h1>
<h2 style="color: #CEDDF4;" >Final Project: Gold and Oil Prediction System</h2>
<h3 style="color: #CEDDF4;" >Python Code</h3>

In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import requests
import pandas as pd
import time
import csv
from itertools import zip_longest
import math
from pathlib import Path
import os

In [2]:
base_dir = Path.cwd()
raw_data_dir=base_dir/'raw data'
cleaned_data_dir=base_dir/'cleaned data'

In [ ]:
#this is the API code for getting the CPI (Consumer Products Index) for Egypt from IMF (International Monetary Fund)
#this API is free

#function to be used at any country with any of published indices of IMF

def imf_data(country,index):
    url=f'https://www.imf.org/external/datamapper/api/v1/{index}/{country}'
    response = requests.get(url)
    data=response.json()
    output=data['values'][index][country]
    df = pd.DataFrame(list(output.items()), columns=['year', index])
    return df

#function to be used for IMF data cleaning 
#cl --> cleaned

def cl_imf_data(dataframe,index,country,start_period,end_period):
    dataframe=dataframe[(dataframe['year']>=start_period)&(dataframe['year']<=end_period)]
    dataframe['year']=pd.to_datetime(dataframe['year'])
    dataframe=dataframe.rename(columns={dataframe.columns[1]:index})
    dataframe.insert(loc=0,column='country',value=country)
    return dataframe

In [ ]:
#get CPI index data for egypt
egy_cpi_df = imf_data('EGY','PCPIPCH')
egy_cpi_df.to_csv(raw_data_dir/"egy_cpi.csv", index=False)

#cleaning of CPI index data for egypt
cl_egy_cpi_df=cl_imf_data(egy_cpi_df,'cpi','egypt','2015','2025')
cl_egy_cpi_df.to_csv(cleaned_data_dir/"egy_cpi.csv", index=False)

In [37]:
#get egyptian data from cbe (central bank of egypt)

#the following options are countermeasure for detecting that we're using code to navigate the website
options=Options()
#options.add_argument("--headless") #To make the code operates in the background
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches",["enable-automation"])
options.add_experimental_option("useAutomationExtension",False)
#option to save the downloaded files into data folders 
prefs= {
    "download.default_directory": str(raw_data_dir),
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "safebrowsing.enabled": True
}
options.add_experimental_option("prefs", prefs)
driver = webdriver.Chrome(options=options)

#function to add the needed data for cbe statistics
## will not work on exchange rates till now
def cbe_data(target_data,file_name):
    cbe_url=f'https://www.cbe.org.eg/en/economic-research/statistics/{target_data.replace(" ","-")}/historical-data'
    driver.get(cbe_url)
    start_date = "01/01/2016"
    driver.execute_script(f"document.getElementById('fromDate').value = '{start_date}';")
    end_date="31/12/2025"
    driver.execute_script(f"document.getElementById('toDate').value = '{end_date}';")
    driver.find_element(By.XPATH,'/html/body/div[1]/section[3]/div[1]/form/div[1]/div/button/div/span[2]').click()
    time.sleep(1)
    driver.find_element(By.XPATH,'//*[@id="historicalDataForm"]/div[2]/button[1]/span').click()
    downloaded_filename = max(raw_data_dir.glob('*'), key=os.path.getmtime)
    downloaded_filename.rename(raw_data_dir / f"{file_name}.csv")
    return file_name

In [38]:
cbe_inflation=cbe_data('inflation rates','egy_cbe_inflation_rate')

In [39]:
def cl_cbe_data(df,country):
    df = pd.read_excel(raw_data_dir / 'egy_cbe_inflation_rate.xlsx')
    df=df.drop(columns=['Unnamed: 3','Unnamed: 4'])
    df=df.rename(columns={df.columns[0]:'date',df.columns[1]:'headline_inflation_yy',df.columns[2]:'core_inflation_yy'})
    df.insert(loc=0,column='country',value=country)
    df['date'] = pd.to_datetime(df['date'], errors='coerce').dt.strftime('%d/%m/%Y')
    df = df.dropna(subset=['date'])
    df = df.reset_index(drop=True)
    return df

In [41]:
cl_cbe_data_df=cl_cbe_data(cbe_inflation,'egypt')
cl_cbe_data_df.to_csv(cleaned_data_dir/"egy_cbe_inflation_rates.csv", index=False)

C:\Users\HMSY\AppData\Local\Temp\ipykernel_26948\3842133201.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['date'] = pd.to_datetime(df['date'], errors='coerce').dt.strftime('%d/%m/%Y')


In [ ]:

df.head()


,country,date,headline_inflation_yy,core_inflation_yy
0,egypt,01/12/2025,12.300%,11.800%
1,egypt,01/11/2025,12.300%,12.500%
2,egypt,01/10/2025,12.500%,12.100%
3,egypt,01/09/2025,11.700%,11.300%
4,egypt,01/08/2025,12.000%,10.700%


In [ ]:

df.head()

,country,date,headline_inflation_yy,core_inflation_yy
0,egypt,2025-12-01,12.300%,11.800%
1,egypt,2025-11-01,12.300%,12.500%
2,egypt,2025-10-01,12.500%,12.100%
3,egypt,2025-09-01,11.700%,11.300%
4,egypt,2025-08-01,12.000%,10.700%
